<a href="https://www.kaggle.com/code/ltrunganhquc/financial-transaction-eda?scriptVersionId=343164715" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **1. Dataset Understanding**

## **1.1 Dataset Loading and Initial Inspection**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

file_path = "/kaggle/input/datasets/ltrunganhquc/raw-dataset/Fraud Detection Dataset.csv"

df = pd.read_csv(file_path)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

### **Dataset Overview Findings**

- The dataset contains 1,000,000 transaction records and 17 columns.
- It includes transaction identifiers, customer/card/device identifiers,
  merchant information, transaction amount, currency, timestamp,
  geographic attributes, and fraud-related labels.
- Most operational transaction fields are fully populated in the raw dataset.
- `fraud_type` is sparsely populated and requires further analysis to determine
  whether its null values are expected or represent data quality issues.
- `timestamp` is initially loaded as an object and requires conversion to
  datetime before temporal validation and analysis.
  

## **1.2 Schema profiling**

In [ ]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

cardinality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_count": df.nunique(dropna=False),
    "unique_pct": (
        df.nunique(dropna=False) / len(df) * 100
    ).round(4)
})

cardinality

In [ ]:
print(df["timestamp"].dtype)
print("Invalid timestamps:", df["timestamp"].isna().sum())
print("Min timestamp:", df["timestamp"].min())
print("Max timestamp:", df["timestamp"].max())

In [ ]:
df[
    [
        "transaction_id",
        "customer_id",
        "card_id",
        "device_id",
        "ip_address",
        "merchant_id",
        "merchant_category",
        "merchant_country",
        "merchant_city",
        "transaction_type",
        "currency",
        "is_fraud",
        "fraud_type"
    ]
].nunique(dropna=False)

### **Schema Profiling Findings**

- `transaction_id` is unique within this dataset and can be used as the transaction-level identifier for subsequent EDA.
- `timestamp` was successfully converted to UTC datetime with no invalid values.
- The dataset covers approximately one year, from 2024-10-30 to 2025-10-30.
- Customer, card, device, IP address, and merchant fields have high cardinality and should be treated primarily as entity identifiers rather than ordinary categorical features.
- `merchant_category`, `merchant_country`, `transaction_type`, `currency`, and fraud-related fields have low cardinality and are suitable for domain and consistency validation.
- `currency` contains only one unique value, indicating that this dataset does not provide meaningful multi-currency variation for reconciliation analysis.

# 2. **Data Quality Profiling**

## **2.1 Missing Value Profiling**

In [ ]:
missing_profile = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(4)
}).sort_values("missing_count", ascending=False)

missing_profile

In [ ]:
pd.crosstab(
    df["is_fraud"],
    df["fraud_type"].isna(),
    margins=True
)

### **Missing Value Findings**

- All core transaction fields are fully populated with no missing values.
- `fraud_type` is missing in 998,510 records (99.851%), but this is expected rather than a data quality issue.
- All non-fraud transactions (`is_fraud = 0`) have `fraud_type = null`.
- All fraud transactions (`is_fraud = 1`) have a populated `fraud_type`.
- Therefore, `fraud_type` behaves as a conditional field whose validity depends on the value of `is_fraud`.

## **2.2 Duplicate Profiling**

In [ ]:
print("Duplicate transaction_id:",
      df["transaction_id"].duplicated().sum())

print("Exact duplicate rows:",
      df.duplicated().sum())

In [ ]:
print("Duplicate customer-card pairs:",
      df[["customer_id", "card_id"]].duplicated().sum())

In [ ]:
df.groupby("transaction_id").size().value_counts().head()

### **Duplicate & Identifier Integrity Findings**

- No duplicate `transaction_id` values were detected across 1,000,000 records.
- No exact duplicate rows were found.
- Every `transaction_id` appears exactly once, confirming strong transaction-level uniqueness in this dataset.
- Repeated `customer_id`–`card_id` pairs are expected because the same customer can legitimately use the same card across multiple transactions.
- Therefore, repeated customer-card pairs should not be treated as duplicate transactions.
- No duplicated transaction identities are present in this dataset, so conflicting-duplicate behavior cannot be empirically evaluated from the raw dataset and must be tested separately using controlled test fixtures.

## **2.3 Domain & Value Validity**

In [ ]:
print("Transaction types:")
print(df["transaction_type"].value_counts())

print("\nCurrencies:")
print(df["currency"].value_counts())

print("\nMerchant categories:")
print(df["merchant_category"].value_counts())

print("\nMerchant countries:")
print(df["merchant_country"].value_counts())

In [ ]:
print("Negative amount:", (df["amount"] < 0).sum())
print("Zero amount:", (df["amount"] == 0).sum())
print("Min amount:", df["amount"].min())
print("Max amount:", df["amount"].max())

In [ ]:
print(
    "Invalid latitude:",
    (~df["merchant_latitude"].between(-90, 90)).sum()
)

print(
    "Invalid longitude:",
    (~df["merchant_longitude"].between(-180, 180)).sum()
)

### Domain & Value Validity Findings

- `transaction_type` contains two observed values: `purchase` and `transfer`.
- The dataset is effectively single-currency, with all transactions recorded in USD.
- Merchant categories and merchant countries contain a limited and consistent set of categorical values.
- No negative or zero transaction amounts were found.
- Transaction amounts range from 1.00 to 5,000.00.
- All merchant latitude and longitude values fall within valid geographic ranges.
- No obvious domain-level validity violations were detected in the tested fields.

## **2.4 Cross-field Consistency Analysis**

In [ ]:
# 1. Fraud label consistency
fraud_consistency = pd.crosstab(
    df["is_fraud"],
    df["fraud_type"].isna()
)

fraud_consistency

In [ ]:
transaction_category = pd.crosstab(
    df["transaction_type"],
    df["merchant_category"]
)

transaction_category

In [ ]:
merchant_consistency = (
    df.groupby("merchant_id")
      .agg(
          category_count=("merchant_category", "nunique"),
          country_count=("merchant_country", "nunique"),
          city_count=("merchant_city", "nunique"),
          latitude_count=("merchant_latitude", "nunique"),
          longitude_count=("merchant_longitude", "nunique")
      )
)

merchant_consistency.describe()

In [ ]:
merchant_conflicts = merchant_consistency[
    (merchant_consistency["category_count"] > 1) |
    (merchant_consistency["country_count"] > 1) |
    (merchant_consistency["city_count"] > 1) |
    (merchant_consistency["latitude_count"] > 1) |
    (merchant_consistency["longitude_count"] > 1)
]

print("Merchant IDs with varying attributes:", len(merchant_conflicts))

In [ ]:
customer_per_card = (
    df.groupby("card_id")["customer_id"]
      .nunique()
)

print(
    "Cards linked to multiple customers:",
    (customer_per_card > 1).sum()
)

customer_per_card.describe()

## **2.4 Cross-field Consistency Findings**

- `is_fraud` and `fraud_type` are fully consistent:
  - all non-fraud transactions have `fraud_type = null`
  - all fraud transactions have a populated `fraud_type`
- `transaction_type` and `merchant_category` are also fully aligned:
  - all `transfer` transactions belong to the `transfer` category
  - purchase transactions are associated only with non-transfer merchant categories
- Merchant category and country remain stable for each `merchant_id`.
- However, 407 merchant IDs are associated with multiple city and/or coordinate values.
- 177 card IDs are linked to multiple customer IDs, with some cards associated with up to 9 customers.
- These two patterns should be treated as consistency candidates for further inspection rather than immediate data quality failures.

## **2.5 Entity Consistency Candidate Inspection**

In [ ]:
merchant_conflicts.sort_values(
    ["city_count", "latitude_count", "longitude_count"],
    ascending=False
).head(10)

In [ ]:
sample_merchant = merchant_conflicts.sort_values(
    ["city_count", "latitude_count"],
    ascending=False
).index[0]

df[df["merchant_id"] == sample_merchant][
    [
        "merchant_id",
        "merchant_category",
        "merchant_country",
        "merchant_city",
        "merchant_latitude",
        "merchant_longitude"
    ]
].drop_duplicates()

In [ ]:
multi_customer_cards = customer_per_card[
    customer_per_card > 1
].sort_values(ascending=False)

multi_customer_cards.head(10)

In [ ]:
sample_card = multi_customer_cards.index[0]

df[df["card_id"] == sample_card][
    [
        "card_id",
        "customer_id",
        "timestamp",
        "transaction_id"
    ]
].sort_values("timestamp").head(30)

### **2.5 Entity Consistency Candidate Inspection Findings**

- Some merchant IDs are associated with multiple cities and geographic coordinates while maintaining the same merchant category and country.
- For example, `MID001487` appears in three different cities with three distinct coordinate pairs.
- This pattern may indicate multi-location merchants, branch-level variation, or synthetic data generation behavior.
- Therefore, merchant location variation should not automatically be classified as a data quality failure without an explicit business rule defining one fixed location per merchant.

- Some card IDs are linked to multiple customer IDs.
- `CARD0000228-1`, for example, is associated with nine different customers over the observed period.
- The card repeatedly appears with `CUST0000228`, while several other customer IDs occur only occasionally.
- This pattern is a stronger entity-integrity anomaly candidate than merchant-location variation.
- However, without a confirmed business rule that each card must belong to exactly one customer, it should be flagged for investigation rather than automatically quarantined.

# **3. Univariate Analysis** 

## **3.1 Transaction Amount Distribution**

In [ ]:
df["amount"].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999
    ]
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df["amount"],
    bins=60
)

plt.xlabel("Transaction Amount (USD)")
plt.ylabel("Frequency")
plt.title("Distribution of Transaction Amount")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    np.log1p(df["amount"]),
    bins=60
)

plt.xlabel("log(1 + Transaction Amount)")
plt.ylabel("Frequency")
plt.title("Log-scaled Transaction Amount Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.boxplot(
    df["amount"],
    vert=False
)

plt.xlabel("Transaction Amount (USD)")
plt.title("Transaction Amount Boxplot")
plt.show()

### 3.1 Transaction Amount Distribution Findings

- Transaction amounts are strongly right-skewed.
- Most transactions are concentrated in the lower-value range, while a relatively small number of transactions extend into a long right tail.
- The boxplot shows a large number of observations beyond the upper whisker, indicating many statistically extreme values relative to the central distribution.
- These high-value observations should not automatically be treated as invalid transactions because they may represent legitimate high-value activity.

## **3.2 Categorical Variable Distributions**

In [ ]:
# Merchant category
df["merchant_category"].value_counts().plot(
    kind="bar",
    figsize=(10, 5),
    title="Transaction Distribution by Merchant Category"
)

plt.xlabel("Merchant Category")
plt.ylabel("Transaction Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Merchant country
df["merchant_country"].value_counts().plot(
    kind="bar",
    figsize=(9, 5),
    title="Transaction Distribution by Merchant Country"
)

plt.xlabel("Merchant Country")
plt.ylabel("Transaction Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Transaction type
df["transaction_type"].value_counts().plot(
    kind="bar",
    figsize=(6, 4),
    title="Transaction Type Distribution"
)

plt.xlabel("Transaction Type")
plt.ylabel("Transaction Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df["fraud_type"].value_counts().plot(
    kind="bar",
    figsize=(8, 4),
    title="Fraud Type Distribution"
)

plt.xlabel("Fraud Type")
plt.ylabel("Transaction Count")
plt.xticks(rotation=45)
plt.show()

### **3.2 Categorical Variable Distribution Findings**

- Most merchant categories have broadly similar transaction volumes, generally around 100,000 transactions.
- `online_marketplace` has substantially fewer transactions than the main merchant categories, while `transfer` is extremely rare.
- The dataset is heavily dominated by `purchase` transactions, with only a very small number of `transfer` transactions.
- Fraud cases are also highly imbalanced across fraud types:
  - `card_testing` is the most common fraud type.
  - `geo_anomaly` is the second most frequent.
  - `account_takeover` occurs less frequently.
  - `money_laundering_ring` is the rarest fraud type.
- These distributions indicate several strongly imbalanced categorical variables, which should be considered when interpreting later fraud-rate and relationship analyses.

## **3.3 Temporal Distribution**

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["date"] = df["timestamp"].dt.date

In [ ]:
hourly_counts = df["hour"].value_counts().sort_index()

hourly_counts.plot(
    kind="line",
    figsize=(10, 5),
    marker="o",
    title="Transaction Volume by Hour of Day"
)

plt.xlabel("Hour")
plt.ylabel("Transaction Count")
plt.xticks(range(0, 24))
plt.show()

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_counts = (
    df["day_of_week"]
    .value_counts()
    .reindex(day_order)
)

day_counts.plot(
    kind="bar",
    figsize=(9, 5),
    title="Transaction Volume by Day of Week"
)

plt.xlabel("Day of Week")
plt.ylabel("Transaction Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
daily_counts = df.groupby("date").size()

daily_counts.plot(
    kind="line",
    figsize=(12, 5),
    title="Daily Transaction Volume"
)

plt.xlabel("Date")
plt.ylabel("Transaction Count")
plt.show()

### **3.3 Temporal Distribution Findings**

- Transaction volume is distributed relatively evenly across the 24 hours of the day, with only small fluctuations between individual hours.
- Transaction activity is also balanced across the days of the week, with no clear weekday or weekend concentration.
- Daily transaction volume remains stable throughout most of the one-year observation period.
- The sharp drops at the beginning and end of the daily series are likely caused by partial boundary days rather than genuine behavioral changes.
- Overall, no strong temporal concentration or seasonality is visible from the univariate transaction-volume analysis.

# **4. Bivariate Analysis**

## **4.1 Amount x Merchant Category**

In [ ]:
amount_by_category = (
    df.groupby("merchant_category")["amount"]
      .agg(["count", "mean", "median", "std", "max"])
      .sort_values("median", ascending=False)
)

amount_by_category

In [ ]:
df.boxplot(
    column="amount",
    by="merchant_category",
    figsize=(12, 6),
    showfliers=False
)

plt.title("Transaction Amount by Merchant Category")
plt.suptitle("")
plt.xlabel("Merchant Category")
plt.ylabel("Transaction Amount (USD)")
plt.xticks(rotation=45)
plt.show()

### **4.1 Amount × Merchant Category Findings**

- Most merchant categories show very similar transaction amount distributions.
- Their median transaction amounts are tightly clustered around 20 USD, while mean amounts are generally around 38 USD.
- Most non-transfer categories also share a similar upper range, with observed maximum values reaching 5,000 USD.
- This suggests that merchant category does not substantially change the typical transaction amount distribution in this dataset.
- The `transfer` category shows a narrower amount range and a slightly higher median (23.79 USD), but it contains only 80 records and is therefore not sufficiently represented for strong conclusions.
- Based on the observed data, merchant-category-specific amount thresholds are not strongly justified. A global amount baseline may be sufficient for monitoring, while category-aware thresholds should only be introduced if production data demonstrates meaningful category-specific differences.

## **4.2 Amount x Transaction Type**

In [ ]:
amount_by_type = (
    df.groupby("transaction_type")["amount"]
      .agg(["count", "mean", "median", "std", "min", "max"])
)

amount_by_type

In [ ]:
amount_percentiles_by_type = (
    df.groupby("transaction_type")["amount"]
      .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
      .unstack()
)

amount_percentiles_by_type

In [ ]:
df.boxplot(
    column="amount",
    by="transaction_type",
    figsize=(7, 5),
    showfliers=False
)

plt.title("Transaction Amount by Transaction Type")
plt.suptitle("")
plt.xlabel("Transaction Type")
plt.ylabel("Transaction Amount (USD)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for tx_type in df["transaction_type"].unique():
    subset = df.loc[
        df["transaction_type"] == tx_type,
        "amount"
    ]

    plt.hist(
        np.log1p(subset),
        bins=40,
        density=True,
        alpha=0.5,
        label=tx_type
    )

plt.xlabel("log(1 + Transaction Amount)")
plt.ylabel("Density")
plt.title("Normalized Amount Distribution by Transaction Type")
plt.legend()
plt.show()

### **4.2 Amount × Transaction Type Findings**

- Purchase transactions dominate the dataset, while transfer transactions are extremely rare (80 records only).
- Purchase transactions have a higher mean amount (38.20) but a slightly lower median amount (20.09) than transfers (23.79).
- Transfer amounts are more tightly concentrated, with a maximum of 100.35 and a 99th percentile of approximately 94.17.
- Purchase transactions show a much heavier right tail, with a 99th percentile of approximately 294.13 and a maximum of 5,000.
- This suggests that transaction type provides useful context for interpreting amount distributions.
- However, because the transfer sample is very small, the observed difference should be treated as exploratory evidence rather than a basis for a hard transaction-type-specific quality rule.

## **4.3 Amount x Time**

In [ ]:
amount_by_hour = (
    df.groupby("hour")["amount"]
      .agg(["count", "mean", "median", "std"])
)

amount_by_hour

In [ ]:
amount_by_hour[["mean", "median"]].plot(
    kind="line",
    figsize=(10, 5),
    marker="o",
    title="Mean and Median Transaction Amount by Hour"
)

plt.xlabel("Hour of Day")
plt.ylabel("Amount (USD)")
plt.xticks(range(24))
plt.show()

In [ ]:
amount_by_day = (
    df.groupby("day_of_week")["amount"]
      .agg(["count", "mean", "median", "std"])
      .reindex(day_order)
)

amount_by_day

In [ ]:
amount_by_day[["mean", "median"]].plot(
    kind="bar",
    figsize=(10, 5),
    title="Mean and Median Transaction Amount by Day of Week"
)

plt.xlabel("Day of Week")
plt.ylabel("Amount (USD)")
plt.xticks(rotation=45)
plt.show()

### 4.3 Amount × Time Findings

- Median transaction amounts remain highly stable across all hours of the day, staying close to 20 USD.
- Mean transaction amounts also vary only slightly by hour, generally remaining around 38 USD.
- No clear hourly period shows a materially different typical transaction amount.
- The same pattern is observed across days of the week, where median amounts remain approximately 20 USD and mean amounts remain close to 38 USD.
- Therefore, time-of-day and day-of-week do not provide strong contextual separation for transaction amount in this dataset.
- Based on the observed data, hour-specific or weekday-specific amount thresholds are not justified.

## **4.4 Optional Fraud Context**

In [ ]:
amount_by_fraud = (
    df.groupby("is_fraud")["amount"]
      .agg(["count", "mean", "median", "std", "max"])
)

amount_by_fraud

In [ ]:
df.boxplot(
    column="amount",
    by="is_fraud",
    figsize=(7, 5),
    showfliers=False
)

plt.title("Transaction Amount by Fraud Status")
plt.suptitle("")
plt.xlabel("Fraud Status")
plt.ylabel("Transaction Amount (USD)")
plt.show()

### **4.4 Optional Fraud Context Findings**

- Fraud transactions show a substantially lower median amount (2.57 USD) than non-fraud transactions (20.10 USD).
- The mean amount is also lower for fraud transactions (24.93 USD vs. 38.22 USD).
- Fraud transactions still include some high-value cases, but their observed maximum (1,794.48 USD) is well below the maximum for non-fraud transactions (5,000 USD).
- Therefore, high transaction amount alone is not a reliable indicator of fraud in this dataset.
- This supports treating amount-based outliers as soft anomaly signals rather than automatic rejection or quarantine conditions.

# **5. Multivariate Analysis**

## **5.1 Merchant Category × Amount × Time**

In [ ]:
category_hour_median = (
    df.groupby(["merchant_category", "hour"])["amount"]
      .median()
      .unstack()
)

category_hour_median

In [ ]:
category_hour_median.T.plot(
    figsize=(12, 6),
    title="Median Transaction Amount by Hour and Merchant Category"
)

plt.xlabel("Hour of Day")
plt.ylabel("Median Amount (USD)")
plt.xticks(range(24))
plt.legend(
    title="Merchant Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.show()

### **5.1 Merchant Category × Amount × Time Findings**

- Median transaction amounts remain close to 20 USD across most merchant categories and hours of the day.
- The non-transfer categories show very stable and similar temporal amount patterns, with only minor hourly fluctuations.
- The `transfer` category shows large fluctuations across hours, including several sharp peaks and missing hourly values.
- However, these fluctuations are driven by the very small transfer sample (80 transactions in total), making hourly transfer medians unstable and unreliable for rule design.
- Overall, combining merchant category with hour-of-day does not reveal meaningful contextual amount regimes for the major transaction categories.
- Therefore, category-and-hour-specific amount thresholds are not supported by this dataset.

## **5.2 Transaction Type x Amount x Time**

In [ ]:
type_hour_profile = (
    df.groupby(["transaction_type", "hour"])
      .agg(
          transaction_count=("transaction_id", "count"),
          median_amount=("amount", "median"),
          mean_amount=("amount", "mean")
      )
      .reset_index()
)

type_hour_profile

In [ ]:
plt.figure(figsize=(10, 5))

for tx_type in type_hour_profile["transaction_type"].unique():
    subset = type_hour_profile[
        type_hour_profile["transaction_type"] == tx_type
    ]

    plt.plot(
        subset["hour"],
        subset["median_amount"],
        marker="o",
        label=tx_type
    )

plt.xlabel("Hour of Day")
plt.ylabel("Median Amount (USD)")
plt.title("Median Transaction Amount by Hour and Transaction Type")
plt.xticks(range(24))
plt.legend()
plt.show()

In [ ]:
transfer_hour_profile = type_hour_profile[
    type_hour_profile["transaction_type"] == "transfer"
]

transfer_hour_profile

### 5.2 Transaction Type × Amount × Time Findings

- Purchase transactions show a highly stable hourly amount pattern, with median values remaining close to 20 USD throughout the day.
- Transfer transactions show large hourly fluctuations in median amount, ranging from approximately 11 USD to 66 USD.
- However, transfer volume is extremely sparse, with most hours containing only 1–8 transfer transactions and some hours containing none.
- Therefore, the apparent hourly volatility of transfer amounts is likely driven by small sample size rather than a reliable temporal pattern.
- The data does not provide sufficient evidence to justify transaction-type-and-hour-specific amount thresholds.
- For quality-gate design, transaction type may provide some contextual information, but temporal segmentation should not be introduced based on this dataset.

## **5.3 Entity Activity Aggreration**

In [ ]:
customer_profile = (
    df.groupby("customer_id")
      .agg(
          transaction_count=("transaction_id", "count"),
          total_amount=("amount", "sum"),
          median_amount=("amount", "median"),
          max_amount=("amount", "max"),
          unique_cards=("card_id", "nunique"),
          unique_devices=("device_id", "nunique"),
          unique_ips=("ip_address", "nunique")
      )
)

customer_profile.describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

In [ ]:
customer_profile.sort_values(
    "transaction_count",
    ascending=False
).head(10)

In [ ]:
customer_profile.sort_values(
    "total_amount",
    ascending=False
).head(10)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    customer_profile["transaction_count"],
    customer_profile["total_amount"],
    alpha=0.3
)

plt.xlabel("Transaction Count")
plt.ylabel("Total Transaction Amount")
plt.title("Customer Transaction Frequency vs Total Amount")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    customer_profile["transaction_count"],
    customer_profile["median_amount"],
    alpha=0.3
)

plt.xlabel("Transaction Count")
plt.ylabel("Median Transaction Amount")
plt.title("Customer Transaction Frequency vs Median Amount")
plt.show()

In [ ]:
customer_profile[
    ["unique_cards", "unique_devices", "unique_ips"]
].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

### 5.3 Entity Activity Aggregation Findings

- Customers have a median of 10 transactions, with 99% having 18 transactions or fewer.
- Customer transaction frequency is relatively concentrated, with a maximum of 27 transactions.
- Total transaction amount varies much more strongly than transaction count.
- Several customers with only a moderate number of transactions accumulate very high total amounts, indicating that high total spending is often driven by unusually large transactions rather than transaction frequency alone.
- Customer median transaction amounts also vary substantially, with some customers showing much higher typical transaction values than the overall population.
- Most customers use approximately 2 cards, 2 devices, and around 10 unique IP addresses.
- A small number of customers show higher entity diversity, reaching up to 6 cards, 9 devices, and 26 IP addresses.
- These extreme entity-level values should be treated as behavioral anomaly candidates rather than deterministic data quality failures.

# **6. Statistical Outlier & Anomaly Analysis**

In [ ]:
q1 = df["amount"].quantile(0.25)
q3 = df["amount"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

amount_outliers = df[
    (df["amount"] < lower_bound) |
    (df["amount"] > upper_bound)
]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outlier rows:", len(amount_outliers))
print(
    "Outlier percentage:",
    round(len(amount_outliers) / len(df) * 100, 4)
)

In [ ]:
df["amount"].quantile([
    0.90,
    0.95,
    0.99,
    0.999
])

In [ ]:
df.nlargest(20, "amount")[
    [
        "transaction_id",
        "timestamp",
        "customer_id",
        "merchant_category",
        "transaction_type",
        "amount",
        "currency"
    ]
]

### 6.1 IQR-based Amount Outlier Findings

- The transaction amount distribution is strongly right-skewed.
- Using the standard 1.5 × IQR rule produces an upper bound of 87.535 USD.
- This threshold flags 87,583 transactions, representing 8.7583% of the dataset.
- The 95th percentile is already 123.04 USD, while the 99th percentile reaches 294.12 USD, showing that a substantial portion of valid observations naturally exists above the IQR threshold.
- The highest observed amounts reach 5,000 USD across multiple merchant categories and customers rather than being isolated to a single context.
- Therefore, the IQR method identifies statistical rarity but does not provide sufficient evidence of semantic invalidity.
- IQR-based amount outliers should be treated as soft anomaly candidates rather than automatic quality-gate rejection conditions.

## **6.2 — Context-aware Outlier Comparison**

In [ ]:
df["amount_outlier_iqr"] = df["amount"] > upper_bound

outlier_by_category = (
    df.groupby("merchant_category")["amount_outlier_iqr"]
      .agg(["count", "sum", "mean"])
)

outlier_by_category["outlier_rate_pct"] = (
    outlier_by_category["mean"] * 100
)

outlier_by_category.sort_values(
    "outlier_rate_pct",
    ascending=False
)

In [ ]:
outlier_by_category["outlier_rate_pct"] \
    .sort_values(ascending=False) \
    .plot(
        kind="bar",
        figsize=(10, 5),
        title="IQR Outlier Rate by Merchant Category"
    )

plt.xlabel("Merchant Category")
plt.ylabel("Outlier Rate (%)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
outlier_by_type = (
    df.groupby("transaction_type")["amount_outlier_iqr"]
      .agg(["count", "sum", "mean"])
)

outlier_by_type["outlier_rate_pct"] = (
    outlier_by_type["mean"] * 100
)

outlier_by_type

### 6.2 Context-aware Outlier Comparison Findings

- IQR-based amount outlier rates are highly consistent across the major merchant categories, generally ranging from approximately 8.6% to 9.0%.
- `online_marketplace` has the highest observed outlier rate at approximately 9.00%, while `restaurants` has the lowest among the major categories at approximately 8.60%.
- The small variation across categories suggests that merchant category does not materially change the behavior of the global IQR threshold in this dataset.
- The `transfer` category has a much lower outlier rate of 2.5%, but this result is based on only 80 transactions and is not sufficiently reliable for threshold design.
- Therefore, category-specific IQR thresholds are not strongly justified by the observed data.
- However, the global IQR rule still flags approximately 8.76% of all transactions, so it remains unsuitable as a hard quality-gate rejection rule.

## **6.3 Hard Error vs Soft Anomaly Classification**

| Observed condition                                          | Interpretation                     | Severity      | Suggested action                            |
| ----------------------------------------------------------- | ---------------------------------- | ------------- | ------------------------------------------- |
| Missing required transaction field                          | Deterministic data-quality failure | Critical      | Quarantine                                  |
| Invalid timestamp parsing                                   | Deterministic schema/value failure | Critical      | Quarantine / reject record                  |
| `amount <= 0`                                               | Invalid transaction amount         | Critical      | Quarantine                                  |
| Unsupported currency                                        | Domain violation                   | Critical      | Quarantine                                  |
| Duplicate `transaction_id` with same payload                | Idempotent duplicate               | Low           | Skip / deduplicate                          |
| Same transaction identity with conflicting immutable fields | Semantic conflict                  | Critical      | Quarantine + review                         |
| `is_fraud` inconsistent with `fraud_type`                   | Cross-field inconsistency          | Medium        | Quarantine / validation failure             |
| Invalid latitude/longitude                                  | Attribute validity failure         | Low–Medium    | Warning or quarantine depending on contract |
| Amount above IQR threshold                                  | Statistical outlier only           | Warning       | Monitor / flag                              |
| High customer card/device/IP diversity                      | Behavioral anomaly candidate       | Warning       | Review / monitoring                         |
| Merchant ID appearing at multiple locations                 | Ambiguous entity relationship      | Informational | Do not reject automatically                 |


### 6.3 Hard Error vs Soft Anomaly Classification Findings

- Deterministic violations such as missing required fields, invalid schema values, unsupported domains, or non-positive transaction amounts are suitable candidates for hard quality-gate rules.
- Statistical rarity alone is not sufficient evidence of data invalidity.
- The IQR method flags approximately 8.76% of transactions, demonstrating that amount outliers should be treated as soft anomaly signals rather than rejection conditions.
- Entity-level patterns such as high card, device, or IP diversity also require contextual interpretation and should remain warning or review candidates unless supported by an explicit business invariant.
- Quality-gate actions should therefore distinguish between deterministic validation failures, semantic conflicts, and soft behavioral anomalies.

# **7. Key EDA Findings**


### 1. **Core transaction fields are structurally complete in this dataset.**
   Required transaction attributes contain no missing values, while `fraud_type` is conditionally populated only for fraudulent transactions and should not be treated as a generic missing-value issue.

### 2. **No raw transaction duplicates were observed.**
   `transaction_id` is unique across the 1,000,000 records and no exact duplicate rows were found. However, conflicting duplicate behavior cannot be evaluated directly from this dataset and requires controlled test fixtures.

### 3. **The amount distribution is strongly right-skewed.**
   Most transactions are relatively small, while a long right tail extends up to 5,000 USD.

### 4. **Statistical amount outliers are common but not necessarily invalid.**
   The standard 1.5 × IQR rule flags approximately 8.76% of transactions, which is too large to justify automatic rejection.

### 5. **Merchant category and time provide limited additional amount segmentation.**
   Major merchant categories show similar amount distributions, and hourly/daily transaction behavior remains relatively stable.

### 6. **Transfer-related patterns are statistically weak because of sparse data.**
   Only 80 transfer transactions are present, making observed transfer-specific fluctuations unreliable for threshold design.

### 7. **Entity-level behavior shows useful anomaly candidates.**
   Most customers exhibit moderate card, device, and IP diversity, while a small number of customers show unusually high entity diversity or transaction values.

### 8. **Observed entity inconsistencies require business context.**
   Some merchant IDs appear across multiple locations and some card IDs are associated with multiple customers, but these patterns cannot be classified as invalid without an explicit business invariant.

### 9. **Fraud context reinforces the distinction between anomaly and data quality.**
   Fraudulent transactions do not simply correspond to unusually high amounts, so transaction-value extremes should not be interpreted as automatic fraud or quality failures.

### 10. **Quality-gate rules should prioritize deterministic violations over statistical rarity.**
    Schema violations, missing required fields, invalid domains, semantic conflicts, and impossible values are stronger quarantine candidates than purely statistical outliers.

# **8. Findings → Data Quality Implications**


| Finding | Evidence | Rule Candidate | Severity | Suggested Action |
|---|---|---|---|---|
| Required transaction fields are complete in the observed dataset | No missing values in core transaction fields | Required fields must not be null | Critical | REJECT / QUARANTINE |
| `fraud_type` is conditionally populated | `fraud_type` exists only when `is_fraud = 1` | Validate `fraud_type` conditionally rather than globally | Medium | Conditional validation |
| Transaction IDs are unique in the dataset | 0 duplicate `transaction_id` values | Enforce transaction identity uniqueness within partner scope | Critical | Deduplicate or quarantine conflict |
| Same-ID conflicting payload cannot be evaluated from raw data | No conflicting duplicates observed | Same transaction identity with different immutable fields should be treated as conflict | Critical | QUARANTINE + review |
| Amount values are strictly positive | No zero or negative amount observed | Reject non-positive amount | Critical | REJECT / QUARANTINE |
| Amount distribution is strongly right-skewed | IQR flags 8.7583% of records | High amount alone should not be considered invalid | Warning | Flag / monitor |
| Merchant categories show similar amount behavior | Major category outlier rates are approximately 8.6–9.0% | Avoid category-specific hard amount thresholds without contract evidence | Informational | No hard rule |
| Temporal amount behavior is stable | Hourly and daily medians remain similar | Avoid hour/day-specific amount rejection thresholds | Informational | Monitoring baseline only |
| Transfer behavior is sparse | Only 80 transfer records | Do not derive hard transfer-specific thresholds from this sample | Informational | Require more data |
| Some cards are linked to multiple customers | 177 cards associated with more than one customer | Validate card-customer relationship only if contract defines one-to-one ownership | Warning | Review / monitoring |
| Some merchants appear in multiple locations | 407 merchant IDs have varying location attributes | Do not reject multi-location merchants without explicit business invariant | Informational | Monitor |
| Customer entity diversity has a long tail | Up to 6 cards, 9 devices and 26 IPs per customer | High entity diversity may indicate behavioral anomaly | Warning | Flag / review |

# 9. Limitations

- The dataset is external and synthetic/public, so its distributions and entity relationships may not fully represent production reconciliation data.

- Only one currency (`USD`) is observed. Therefore, this EDA cannot evaluate multi-currency validation, currency-specific amount behavior, or FX-related consistency.

- No duplicate `transaction_id` values or same-ID conflicting payloads are present in the raw dataset. Duplicate and immutable-field conflict handling must therefore be validated using controlled test fixtures rather than inferred from this EDA.

- The observed merchant categories, transaction types, and countries represent dataset-specific values and should not be treated as the production allowlist. Production domain validation should be driven by partner contracts or configuration.

- Transfer transactions are extremely underrepresented, with only 80 records, so transfer-specific statistical conclusions are unreliable.

- Statistical thresholds such as IQR or percentiles are descriptive properties of this dataset and should not be directly converted into production rejection thresholds.

- Entity relationships such as card-to-customer or merchant-to-location mappings cannot be classified as valid or invalid without explicit business invariants.

- Fraud-related fields are used only as supporting analytical context. This notebook is intended for data quality and pipeline validation, not fraud prediction or fraud-model development.

- Temporal behavior reflects the observed one-year dataset window and should be treated as a monitoring baseline rather than a permanent production expectation.

- Final production Quality Gate rules should combine EDA evidence with partner schema contracts, domain rules, operational requirements, and controlled validation scenarios.